# Assignment 04: Blood Pressure Follow-Up

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python3 check_assignment.py` in the terminal.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)

DATA_PATH = Path("data") / "bp_followup.csv"
HOME_PATH = Path("data") / "home_bp.csv"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
LOADED_PATH = OUTPUT_DIR / "bp_loaded.csv"
SUMMARY_PATH = OUTPUT_DIR / "visit_summary.csv"
COUNTS_PATH = OUTPUT_DIR / "clinic_counts.csv"
FOLLOWUP_PATH = OUTPUT_DIR / "followup_priority.csv"
PARQUET_PATH = OUTPUT_DIR / "followup_priority.parquet"
GAP_PATH = OUTPUT_DIR / "white_coat_gap.csv"

print("data files found:", DATA_PATH.exists() and HOME_PATH.exists())

## Task 1: Put the cells in running order

### 1.1 Reorder the cells

The next two code cells are out of order: the first uses `vials_on_hand`, which the second defines. Move the whole producer cell above the dependent cell. Do not copy the definition into another cell.

In [ ]:
vials_on_hand = 3
print("vials_on_hand:", vials_on_hand)

In [ ]:
doses_available = vials_on_hand * 10
print("doses_available:", doses_available)

### 1.2 Explain the repair

Explain the difference between the order the cells appear in and the order they ran, why the first cell can print a result even while it sits above the cell it needs, why output saved under a cell is not proof that the notebook works now, and what you changed and how Restart and Run All shows that it worked.

Your explanation here. Include the word periwinkle.

Explanation:
In a Jupyter notebook, cells appear in a particular order from top to bottom. However, the order in which they appear is different from the order they ran because cells can be run independently from each other and doesn't necessarily need to be performed how they are seen. For instance, the second cell can be run before the first cell if there is no conflict.

Although it is sitting above the cell it needs, the first cell can print a result because the bottom cell is separate and can be run independently. We can run the bottom cell first, storing the variable we need, and then run the first cell without errors. 

The output saved under a cell is not proof of the notebook working as running cells independently and out of order can change or overwrite the state of our variables and lead to unexpected results in later operations and sessions, like periwinkles quickly spreading from variable to variable and forming dense foliage (abnormal value and error problems) that needs to be cleared. 

To ensure my code runs in the proper order, I moved the cell containing "vials_on_hand" above the cell with "doses_available." I then pressed "Restart" to clear my variables and pressed "Run All" to perform my code with a clean slate. Doing a Restart and Run All shows my notebook works because it resets all variables and stored values, allowing us to run our code with a blank slate (no underlying bugs or unexpected variables) and confirm it runs without error in our expected order.

## Task 2: Load the blood pressure export

### 2.1 Look at the raw file

Run this cell as it is: it prints the file and reads it with no options.

In [ ]:
print(open(DATA_PATH).read())
raw = pd.read_csv(DATA_PATH)
print("shape:", raw.shape)
raw.head(3)

### 2.2 Read with options and save

> **Checkpoint: `output/bp_loaded.csv`**
> 14 rows under the header `patient_id,clinic,age,sbp_baseline,sbp_week4,sbp_week8`.

In [ ]:
# TODO: read DATA_PATH with sep, skiprows, usecols, na_values, and index_col
bp = pd.read_csv(DATA_PATH, sep = ";", skiprows = [1], usecols = ["patient_id", "clinic", "age", "sbp_baseline", "sbp_week4", "sbp_week8"], na_values = ["-999"], index_col = "patient_id")  

print("shape:", bp.shape)
print(bp.dtypes)
display(bp.head())

# TODO: display patient P104's row with .loc, then the first three rows with .iloc
display(bp.loc["P104"])

display(bp.iloc[0:3])

# TODO: save bp to LOADED_PATH, keeping its patient_id index
bp.to_csv(LOADED_PATH, index=True)


## Task 3: Summarize the readings

### 3.1 Summarize each visit

> **Checkpoint: `output/visit_summary.csv`**
> Three rows, `sbp_baseline`, `sbp_week4`, and `sbp_week8`, under the header `visit,mean,median,count`.

In [ ]:
readings = bp[["sbp_baseline", "sbp_week4", "sbp_week8"]]


#visit_summary = pd.DataFrame(readings.mean(), index = ["sbp_baseline", "sbp_week4", "sbp_week8"], name = "visit")  
visit_summary = pd.DataFrame({"mean": readings.mean(), "median": readings.median(), "count": readings.count()}, index = ["sbp_baseline", "sbp_week4", "sbp_week8"])
# TODO: a DataFrame with the columns mean, median, and count, one row per visit
# TODO: name the index visit
visit_summary.index.name = "visit"
display(round(visit_summary, 2))


highest_baseline = readings["sbp_baseline"].idxmax()  # TODO: the patient_id with the highest sbp_baseline
baseline_week8_r = readings["sbp_baseline"].corr(readings["sbp_week8"])  # TODO: the correlation of sbp_baseline with sbp_week8
print("highest baseline:", highest_baseline)
print("baseline vs week 8 r:", baseline_week8_r)

# TODO: save visit_summary to SUMMARY_PATH, keeping its visit index
visit_summary.to_csv(SUMMARY_PATH, index=True)

### 3.2 Count patients per clinic

> **Checkpoint: `output/clinic_counts.csv`**
> Four rows under the header `clinic,count`, North first.

In [ ]:
clinic_counts = bp["clinic"].value_counts()  # TODO: the number of patients at each clinic, with value_counts()
display(clinic_counts)
print("clinics:", clinic_counts.nunique())  # TODO: replace None with the number of distinct clinics, from nunique()

# TODO: save clinic_counts to COUNTS_PATH
clinic_counts.to_csv(COUNTS_PATH)

## Task 4: Build the follow-up list

### 4.1 Select the program patients

In [ ]:
sbp_filter = bp.loc[bp["sbp_baseline"] >= 140]
in_program = sbp_filter.isin(["North", "East"])  # TODO: True for North or East patients (isin) whose sbp_baseline is 140 or more
followup = sbp_filter[in_program["clinic"]].drop(columns = "age")  # TODO: the in_program rows of bp, without the age column

print("program patients:", in_program["clinic"].sum())
followup

### 4.2 Add the derived columns

In [ ]:
# TODO: add sbp_mean, the mean of each patient's three readings (axis="columns")
sbp_readings = followup[["sbp_baseline", "sbp_week4", "sbp_week8"]]
followup["sbp_mean"] = round(sbp_readings.mean(axis="columns"), 2)

# TODO: add change_week8, sbp_week8 minus sbp_baseline
followup["change_week8"] = followup["sbp_week8"]-followup["sbp_baseline"]

followup

### 4.3 Sort, rank, and save

> **Checkpoint: `output/followup_priority.csv` and `output/followup_priority.parquet`**
> Seven patients, P110 first and P107 last, with the columns `README.md` lists.

In [ ]:
# TODO: sort followup by change_week8, then patient_id, and assign the result back to followup
followup = followup.sort_values(by = ["change_week8", "patient_id"])

# TODO: add improvement_rank, the rank of change_week8 with method="min"
followup["improvement_rank"] = followup["change_week8"].rank(method = "min")
display(followup)

# TODO: save followup to FOLLOWUP_PATH, keeping its patient_id index
followup.to_csv(FOLLOWUP_PATH, index=True)

# TODO: save followup to PARQUET_PATH with to_parquet(), keeping its index (no index=False)
followup.to_parquet(PARQUET_PATH, index=True)

back = pd.read_parquet(PARQUET_PATH)  # TODO: read PARQUET_PATH back with pd.read_parquet()
print("Parquet round trip matches:", back.equals(followup))

## Task 5: Compare clinic and home readings

> **Checkpoint: `output/white_coat_gap.csv`**
> 15 rows under the header `patient_id,white_coat_gap_mmhg`, five of them with a number.

In [ ]:
home = pd.read_csv(HOME_PATH, index_col = "patient_id") 
# TODO: read HOME_PATH with patient_id as the index

white_coat_gap = bp["sbp_week8"] - home["home_sbp_week8"]  # TODO: bp's sbp_week8 minus home's home_sbp_week8
# TODO: name the Series white_coat_gap_mmhg
white_coat_gap_mmhg = pd.Series(white_coat_gap)

print("patients with both readings:", white_coat_gap.count())

# TODO: save white_coat_gap to GAP_PATH, keeping its patient_id index
white_coat_gap.to_csv(GAP_PATH, index=True)

## Fresh-run check

Click **Restart**, then **Run All**. The last cell prints your score and what to fix.

In [ ]:
from check_assignment import run_checks

run_checks()